In [ ]:
!pip install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile, os

zip_path  = '/content/drive/MyDrive/Archery/yolo_data.zip'
dest_path = '/content/drive/MyDrive/Archery'

if not os.path.exists(os.path.join(dest_path, 'yolo_data')):
    print('Unzipping yolo_data.zip...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dest_path)
    print('Done.')
else:
    print('yolo_data already unzipped, skipping.')

In [ ]:
import os
import sys
from ultralytics import YOLO

BASE = '/content/drive/MyDrive/Archery'
DATA = os.path.join(BASE, 'yolo_data', 'dataset.yaml')
CHECKPOINT = 'yolov8s-pose.pt'  #always start from base — old best.pt was trained with broken 0.30 boxes
PROJECT = os.path.join(BASE, 'runs', 'pose')

EPOCHS = 150
IMGSZ = 1280
BATCH = 8
PATIENCE = 30
RUN_NAME = 'archery'

In [ ]:
model = YOLO(CHECKPOINT)
model.train(
    data       = DATA,
    epochs     = EPOCHS,
    imgsz      = IMGSZ,
    batch      = BATCH,
    project    = PROJECT,
    name       = RUN_NAME,
    exist_ok   = False,
    freeze     = 0,
    cache      = 'ram',
    degrees    = 15.0,
    scale      = 0.9,
    hsv_h      = 0.08,
    hsv_s      = 0.7,
    hsv_v      = 0.5,
    perspective= 0.008,
    translate  = 0.1,
    shear      = 5.0,
    mosaic     = 0.0,
    mixup      = 0.0,
    fliplr     = 0.5,
    flipud     = 0.0,
    copy_paste = 0.1,       
    patience   = PATIENCE,
    save       = True,
    plots      = True,
    val        = True,
    verbose    = True,
)

In [ ]:
best_ckpt = os.path.join(PROJECT, RUN_NAME, 'weights', 'best.pt')
best_model = YOLO(best_ckpt)
metrics    = best_model.val(data=DATA, split='test', imgsz=IMGSZ, batch=BATCH)
#box is the bounding box accuracy + pose is the actual arrow point
print(f"mAP50   (box)   : {metrics.box.map50:.4f}")
print(f"mAP50-95 (box)  : {metrics.box.map:.4f}")
if hasattr(metrics, 'pose'):
    print(f"mAP50   (pose)  : {metrics.pose.map50:.4f}")
    print(f"mAP50-95 (pose) : {metrics.pose.map:.4f}")

In [ ]:
#Export best.pt to CoreML (.mlpackage) for the iOS app
#nms=False — the app runs its own NMS in ArrowDetector.swift since it doesn't work with pose
#output saved alongside best.pt in the weights folder
export_model = YOLO(best_ckpt)
export_model.export(
    format = 'coreml',
    imgsz  = IMGSZ,
    nms    = False,
)
mlpackage_path = best_ckpt.replace('.pt', '.mlpackage')
print(f"Exported to: {mlpackage_path}")
print('Replace the ml package inside the swift app now')